In [ ]:
import numpy as np, pandas as pd, os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Experiment 8b — Controlled-Duration Bottleneck Compositionality

Tests whether the Exp 8 headline finding (k=2 compositionality peak) is a real bottleneck effect or a training-duration artifact.

In [ ]:
!pip install sentence-transformers torch matplotlib -q

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

model = SentenceTransformer('LaBSE', device='cpu')
print('LaBSE loaded')

CONCEPTS = [
    'water','fire','earth','sky','love','fear','trust','light','dark','time',
    'life','death','eat','sleep','run','give','take','speak','think','feel',
    'wind','stone','river','mountain','forest','child','elder','friend','enemy','peace',
    'war','pain','joy','hope','dream','build','break','find','lose','change',
]
ANALOGIES = [
    ('love','fear','joy','pain'), ('life','death','peace','war'),
    ('give','take','build','break'), ('light','dark','hope','fear'),
    ('friend','enemy','peace','war'), ('find','lose','give','take'),
    ('build','break','give','take'), ('hope','fear','joy','pain'),
]
CONCEPT_EMBS = model.encode(CONCEPTS, normalize_embeddings=True)
print(f'Concepts: {len(CONCEPTS)}')

class BottleneckAE(nn.Module):
    def __init__(self, k, hidden=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(768,hidden), nn.GELU(), nn.Linear(hidden,hidden//2), nn.GELU(), nn.Linear(hidden//2,k))
        self.decoder = nn.Sequential(
            nn.Linear(k,hidden//2), nn.GELU(), nn.Linear(hidden//2,hidden), nn.GELU(), nn.Linear(hidden,768))
    def forward(self, x):
        z = self.encoder(x)
        return F.normalize(self.decoder(z), dim=-1), z

def train_ae(k, stop_mode='fixed', max_epochs=3000, target_loss=None,
             plateau_patience=200, plateau_eps=1e-5):
    X = torch.tensor(CONCEPT_EMBS, dtype=torch.float32)
    m = BottleneckAE(k); opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    best_loss = float('inf'); plateau = 0; final_loss = None
    for epoch in range(max_epochs):
        m.train()
        recon, z = m(X)
        recon_loss = (1.0 - (recon*X).sum(-1)).mean()
        ref_loss = F.cross_entropy(recon @ X.T, torch.arange(len(X)))
        loss = recon_loss + ref_loss
        opt.zero_grad(); loss.backward(); opt.step()
        final_loss = recon_loss.item()
        if stop_mode == 'converge':
            if final_loss < best_loss - plateau_eps: best_loss=final_loss; plateau=0
            else:
                plateau += 1
                if plateau >= plateau_patience: return m, epoch+1, final_loss
        elif stop_mode == 'matched' and target_loss is not None:
            if final_loss <= target_loss: return m, epoch+1, final_loss
    return m, max_epochs, final_loss

def compositionality_mrr(m):
    X = torch.tensor(CONCEPT_EMBS, dtype=torch.float32)
    m.eval()
    with torch.no_grad():
        Z = m.encoder(X).numpy()
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True)+1e-9)
    rranks = []
    for a,b,c,d in ANALOGIES:
        ia,ib,ic,id_ = [CONCEPTS.index(x) for x in (a,b,c,d)]
        pred = Z[ia]-Z[ib]+Z[ic]; pred /= np.linalg.norm(pred)+1e-9
        sims = Z @ pred
        for x in (ia,ib,ic): sims[x] = -np.inf
        rank = 1+int((sims>sims[id_]).sum())
        rranks.append(1.0/rank)
    return float(np.mean(rranks))

def raw_mrr():
    Z = CONCEPT_EMBS / (np.linalg.norm(CONCEPT_EMBS,axis=1,keepdims=True)+1e-9)
    rranks = []
    for a,b,c,d in ANALOGIES:
        ia,ib,ic,id_ = [CONCEPTS.index(x) for x in (a,b,c,d)]
        pred = Z[ia]-Z[ib]+Z[ic]; pred /= np.linalg.norm(pred)+1e-9
        sims = Z @ pred
        for x in (ia,ib,ic): sims[x] = -np.inf
        rranks.append(1.0/(1+int((sims>sims[id_]).sum())))
    return float(np.mean(rranks))

RAW_MRR = raw_mrr()
K_VALUES = [2, 4, 8, 16, 32, 64, 128]
print(f'Raw LaBSE MRR: {RAW_MRR:.3f}')

In [ ]:
print('=== Condition 1: FIXED 1500 epochs ===')
fixed_results = []
for k in K_VALUES:
    m, ep, fl = train_ae(k, 'fixed', 1500)
    mrr = compositionality_mrr(m)
    fixed_results.append({'k':k,'mrr':mrr,'epochs':ep,'loss':fl})
    print(f'  k={k:>3}: MRR={mrr:.3f}  loss={fl:.4f}')

print('\n=== Condition 2: TRAIN-TO-CONVERGENCE ===')
conv_results = []
for k in K_VALUES:
    m, ep, fl = train_ae(k, 'converge', 5000)
    mrr = compositionality_mrr(m)
    conv_results.append({'k':k,'mrr':mrr,'epochs':ep,'loss':fl})
    print(f'  k={k:>3}: MRR={mrr:.3f}  loss={fl:.4f}  epochs={ep}')

target = max(r['loss'] for r in conv_results)
print(f'\n=== Condition 3: MATCHED-LOSS (target={target:.4f}) ===')
matched_results = []
for k in K_VALUES:
    m, ep, fl = train_ae(k, 'matched', 5000, target_loss=target)
    mrr = compositionality_mrr(m)
    matched_results.append({'k':k,'mrr':mrr,'epochs':ep,'loss':fl})
    print(f'  k={k:>3}: MRR={mrr:.3f}  loss={fl:.4f}  epochs={ep}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

ax = axes[0]
ax.plot(K_VALUES, [r['mrr'] for r in fixed_results],   'o-', label='Fixed epochs (orig)', color='#378ADD', linewidth=2)
ax.plot(K_VALUES, [r['mrr'] for r in conv_results],    's-', label='Train to convergence', color='#1D9E75', linewidth=2)
ax.plot(K_VALUES, [r['mrr'] for r in matched_results], '^-', label='Matched recon loss',   color='#E24B4A', linewidth=2)
ax.axhline(RAW_MRR, color='gray', linestyle=':', label=f'Raw LaBSE ({RAW_MRR:.3f})')
ax.set_xscale('log', base=2)
ax.set_xlabel('Bottleneck k'); ax.set_ylabel('Compositionality (analogy MRR)')
ax.set_title('Does the k=2 compositionality peak survive\ntraining-duration controls?')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(K_VALUES, [r['epochs'] for r in conv_results],    's-', label='convergence',   color='#1D9E75')
ax.plot(K_VALUES, [r['epochs'] for r in matched_results], '^-', label='matched loss',  color='#E24B4A')
ax.set_xscale('log', base=2)
ax.set_xlabel('Bottleneck k'); ax.set_ylabel('Epochs to stop')
ax.set_title('Training duration per k\n(smaller k converges faster = the confound)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Exp 8b — Controlled-Duration Bottleneck Compositionality\n'
             'Is the k=2 peak a real bottleneck effect or a training-duration artifact?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp8b_controlled_compositionality.png', dpi=150, bbox_inches='tight')
plt.show()

def k2_peak(res): return int(np.argmax([r['mrr'] for r in res])) == 0
print(f'k=2 is peak under: fixed={k2_peak(fixed_results)}  convergence={k2_peak(conv_results)}  matched={k2_peak(matched_results)}')
if k2_peak(conv_results) and k2_peak(matched_results):
    print('VERDICT: REAL BOTTLENECK EFFECT — k=2 peak survives training-duration controls.')
elif k2_peak(fixed_results) and not k2_peak(conv_results):
    print('VERDICT: TRAINING ARTIFACT — k=2 peak disappears once duration is controlled.')
else:
    print('VERDICT: MIXED — fragile, needs more seeds.')